# SIBDA-1380 – Avance 1: Formulación, Ingesta, Diagnóstico de Calidad, ETL y Almacenamiento
**Proyecto:** Analítica de Benefactores y Donaciones – Fundación 1380  
**Asignatura:** ET0155 – Fundamentos de Big Data  
**Entregable:** Avance 1 (Google Colab ejecutable)

### Contenido de este Notebook
1. Configuración de entorno y estructura de **Data Lake / Data Warehouse**.
2. **Carga segura e inmutable** del archivo Excel (sin modificar jamás la fuente original; verificación con SHA-256).
3. **Inspección de hojas y columnas** (mapeo de las 130 columnas y 2.744 registros).
4. **Primer Diagnóstico de Calidad de Datos** (completitud, tipos mixtos, anomalías, duplicados, teléfonos compartidos e inconsistencias contables).
5. **Primer Proceso ETL en Python** (normalización, limpieza categórica, parseo híbrido de fechas y *unpivot* histórico 2008–2026).
6. **Persistencia inicial** en **Data Lake (Parquet)** y **Data Warehouse (DuckDB)** con consultas SQL de validación.

In [ ]:
# ==============================================================================
# 1. INSTALACIÓN DE DEPENDENCIAS Y CONFIGURACIÓN INICIAL
# ==============================================================================
!pip install -q duckdb openpyxl pyarrow plotly

import os
import re
import hashlib
import shutil
from datetime import datetime
from pathlib import Path

import numpy as np
import pandas as pd
import openpyxl
import duckdb
import matplotlib.pyplot as plt
import plotly.express as px

pd.set_option('display.max_columns', 50)
pd.set_option('display.max_rows', 100)
print("Entorno configurado correctamente. Pandas:", pd.__version__, "| DuckDB:", duckdb.__version__)

## 2. Arquitectura de Carpetas y Carga Segura del Excel (Sin Modificar el Original)
Se crea la estructura local del **Data Lake** (aw, ronze, silver) y del **Data Warehouse** (warehouse).  
El notebook soporta:
- Lectura desde **Google Drive** (ajustando RUTA_EXCEL_DRIVE), o
- Subida directa desde el computador mediante el selector de **Google Colab** (iles.upload()), o
- Lectura local si el archivo ya está en el directorio actual.

En todos los casos se calcula el **hash SHA-256** del archivo origen y se trabaja únicamente sobre una **copia de trabajo** en datalake/raw/.

In [ ]:
# ==============================================================================
# 2. CARGA SEGURA DEL EXCEL Y VERIFICACIÓN DE INMUTABILIDAD
# ==============================================================================
BASE_DIR = Path("sibda_1380")
RAW_DIR = BASE_DIR / "datalake" / "raw"
BRONZE_DIR = BASE_DIR / "datalake" / "bronze"
SILVER_DIR = BASE_DIR / "datalake" / "silver"
DW_DIR = BASE_DIR / "warehouse"
LOGS_DIR = BASE_DIR / "logs"

for d in [RAW_DIR, BRONZE_DIR, SILVER_DIR, DW_DIR, LOGS_DIR]:
    d.mkdir(parents=True, exist_ok=True)

NOMBRE_ESPERADO = "BD_F1380_SeudonimizadaAnonimizadaencriptada.xlsx"
RUTA_EXCEL_DRIVE = f"/content/drive/MyDrive/Fundacion1380/{NOMBRE_ESPERADO}"

def sha256_file(path: Path) -> str:
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for chunk in iter(lambda: f.read(8192), b""):
            h.update(chunk)
    return h.hexdigest()

origen_path = None
posibles_rutas = [
    Path(NOMBRE_ESPERADO),
    Path("Recursos") / NOMBRE_ESPERADO,
    Path(RUTA_EXCEL_DRIVE),
]

for p in posibles_rutas:
    if p.exists():
        origen_path = p
        break

if origen_path is None:
    try:
        from google.colab import files
        print(f"No se encontró '{NOMBRE_ESPERADO}' en rutas locales/Drive. Sube el archivo Excel:")
        uploaded = files.upload()
        if not uploaded:
            raise FileNotFoundError("No se seleccionó ningún archivo.")
        nombre_subido = list(uploaded.keys())[0]
        origen_path = Path(nombre_subido)
    except ImportError:
        raise FileNotFoundError(
            f"Coloca '{NOMBRE_ESPERADO}' en el directorio de trabajo o ajusta RUTA_EXCEL_DRIVE."
        )

hash_original = sha256_file(origen_path)
timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
copia_raw_path = RAW_DIR / f"raw_{timestamp}_{origen_path.name}"
shutil.copy2(origen_path, copia_raw_path)
hash_copia = sha256_file(copia_raw_path)

assert hash_original == hash_copia, "Error de integridad: el hash de la copia no coincide con el original."

print("Archivo origen :", origen_path)
print("Copia inmutable:", copia_raw_path)
print("SHA-256        :", hash_original)
print("Integridad verificada: el archivo original NO será modificado.")

## 3. Inspección de Hojas, Dimensiones y Estructura de Columnas
Se inspecciona el libro con openpyxl en modo ead_only=True para verificar hojas disponibles, dimensiones, presencia de fórmulas y el inventario completo de columnas agrupadas por bloque funcional.

In [ ]:
# ==============================================================================
# 3. INSPECCIÓN ESTRUCTURAL DEL EXCEL (OPENPYXL READ-ONLY)
# ==============================================================================
wb = openpyxl.load_workbook(copia_raw_path, read_only=True, data_only=False)
print("Hojas disponibles:", wb.sheetnames)

sheet_name = wb.sheetnames[0]
ws = wb[sheet_name]
print(f"Hoja activa: '{sheet_name}' | Filas: {ws.max_row} | Columnas: {ws.max_column}")

# Leer encabezados exactos (fila 1) y verificar si existen fórmulas en una muestra/total
headers_raw = [cell.value for cell in next(ws.iter_rows(min_row=1, max_row=1))]
wb.close()

# Cargar con Pandas manteniendo todas las columnas en crudo (dtype=object) para auditoría fiel
df_raw = pd.read_excel(copia_raw_path, sheet_name=sheet_name, dtype=object)
print(f"DataFrame cargado: {df_raw.shape[0]} registros x {df_raw.shape[1]} columnas")

# Inventario de columnas
col_inventory = pd.DataFrame({
    "indice": range(len(headers_raw)),
    "columna_excel": [openpyxl.utils.get_column_letter(i + 1) for i in range(len(headers_raw))],
    "nombre_original_header": headers_raw,
    "nombre_pandas": df_raw.columns.tolist()
})

def clasificar_bloque(idx: int) -> str:
    if idx <= 14:
        return "1. Identificación y Demografía (A-O)"
    elif idx <= 31:
        return "2. Gestión CRM y Seguimiento 2026 (P-AF)"
    elif idx <= 43:
        return "3. Logística, Preferencias y Recursos (AG-AR)"
    elif idx <= 59:
        return "4. Totales Anuales 2008-2023 (AS-BH)"
    elif idx <= 83:
        return "5. Detalle Mensual + Total 2024 (BI-CF)"
    elif idx <= 106:
        return "6. Detalle Mensual + Total 2025 (CG-DC)"
    else:
        return "7. Detalle Mensual + Total 2026 + Acumulado (DD-DZ)"

col_inventory["bloque"] = col_inventory["indice"].apply(clasificar_bloque)
display(col_inventory.groupby("bloque")["indice"].count().rename("cantidad_columnas").to_frame())
display(col_inventory.head(20))

## 4. Primer Diagnóstico de Calidad de los Datos
En esta sección se cuantifican los problemas de calidad reales presentes en la base de datos:
- **Completitud:** porcentaje de nulos por columna.
- **Validez de tipos:** mezcla de seriales numéricos de Excel, cadenas de fechas malformadas y texto libre en columnas de fechas o montos.
- **Consistencia categórica:** fragmentación de SOCIO, espacios residuales y códigos inválidos en TIPO PAGO.
- **Unicidad e identidad:** duplicados de CEDULA y NOMBRES APELLIDOS, además de la detección de **teléfonos compartidos** entre múltiples benefactores.
- **Consistencia contable:** diferencias entre columnas resumen anuales y la suma de sus columnas mensuales.

In [ ]:
# ==============================================================================
# 4. DIAGNÓSTICO DE CALIDAD DE DATOS
# ==============================================================================
n_rows = len(df_raw)

# 4.1 Completitud y tipos mezclados por columna
diag_rows = []
for idx, col in enumerate(df_raw.columns):
    s = df_raw[col]
    non_null = int(s.notna().sum())
    nulls = int(s.isna().sum())
    pct_null = round(nulls / n_rows * 100, 2)
    tipos = s.dropna().map(lambda x: type(x).__name__).value_counts().to_dict()
    diag_rows.append({
        "indice": idx,
        "columna": col,
        "no_nulos": non_null,
        "nulos": nulls,
        "pct_nulos": pct_null,
        "tipos_detectados": str(tipos),
        "cantidad_tipos": len(tipos),
        "unicos": int(s.nunique(dropna=True))
    })

df_diag = pd.DataFrame(diag_rows)
df_diag.to_csv(LOGS_DIR / "diagnostico_calidad_avance1.csv", index=False)

print("=== TOP 15 COLUMNAS CLAVE CON VALORES FALTANTES (BLOQUES 1-3) ===")
display(df_diag[df_diag["indice"] <= 43].sort_values("pct_nulos", ascending=False).head(15))

print("=== COLUMNAS CON TIPOS MIXTOS (ej. fechas/montos mezclados con texto) ===")
display(df_diag[df_diag["cantidad_tipos"] > 1][["indice", "columna", "no_nulos", "tipos_detectados"]])

# 4.2 Anomalías de fechas en FECHA (DD-MM-AAA) y ULT  PAGO
col_fecha = df_raw.columns[1]
col_ult_pago = df_raw.columns[42]

fechas_texto = df_raw[col_fecha].dropna()
fechas_texto = fechas_texto[fechas_texto.map(lambda x: isinstance(x, str))]
print(f"\n'{col_fecha}' -> valores texto anómalos ({len(fechas_texto)}):", fechas_texto.unique()[:12].tolist())

ult_texto = df_raw[col_ult_pago].dropna()
ult_texto = ult_texto[ult_texto.map(lambda x: isinstance(x, str))]
print(f"'{col_ult_pago}' -> valores texto anómalos ({len(ult_texto)}):", ult_texto.unique()[:12].tolist())

# 4.3 Cardinalidad y ruido en SOCIO y TIPO PAGO
col_socio = df_raw.columns[2]
col_tipo_pago = df_raw.columns[33]
print(f"\n'{col_socio}' -> valores únicos crudos: {df_raw[col_socio].nunique(dropna=True)}")
print("Top 15 valores crudos en SOCIO:")
display(df_raw[col_socio].value_counts(dropna=False).head(15).to_frame("conteo"))

print(f"\n'{col_tipo_pago}' -> distribución cruda:")
display(df_raw[col_tipo_pago].value_counts(dropna=False).to_frame("conteo"))

# 4.4 Duplicados y Teléfonos Compartidos
def resumir_duplicados(serie: pd.Series, nombre: str):
    s = serie.dropna().astype(str).str.strip()
    vc = s.value_counts()
    grupos_rep = vc[vc > 1]
    print(f"{nombre:20s} | No nulos: {len(s):4d} | Únicos: {len(vc):4d} | Grupos repetidos: {len(grupos_rep):3d} | Máx repeticiones: {vc.max() if len(vc) else 0}")
    return grupos_rep

print("\n=== ANÁLISIS DE UNICIDAD Y TELÉFONOS COMPARTIDOS ===")
_ = resumir_duplicados(df_raw[df_raw.columns[0]], "COD")
ced_dups = resumir_duplicados(df_raw[df_raw.columns[7]], "CEDULA")
nom_dups = resumir_duplicados(df_raw[df_raw.columns[4]], "NOMBRES APELLIDOS")
tel_dups = resumir_duplicados(df_raw[df_raw.columns[8]], "TELEFONOS")

print("\nDistribución de personas por teléfono compartido:")
display(tel_dups.value_counts().sort_index().rename_axis("personas_con_mismo_tel").to_frame("cantidad_telefonos"))

In [ ]:
# ==============================================================================
# 4.5 VISUALIZACIONES DEL PRIMER DIAGNÓSTICO DE CALIDAD
# ==============================================================================
fig, axes = plt.subplots(1, 2, figsize=(15, 5))

# Gráfica 1: Top 12 columnas operativas/demográficas con mayor % de nulos
top_nulls = df_diag[df_diag["indice"] <= 43].sort_values("pct_nulos", ascending=True).tail(12)
axes[0].barh(top_nulls["columna"], top_nulls["pct_nulos"], color="#015583")
axes[0].set_title("Top 12 Columnas Operativas con Datos Faltantes (%)")
axes[0].set_xlabel("% de registros nulos")
axes[0].grid(axis="x", alpha=0.3)

# Gráfica 2: Teléfonos compartidos (cantidad de personas por número telefónico)
tel_counts = df_raw[df_raw.columns[8]].dropna().astype(str).str.strip().value_counts()
dist_tel = tel_counts.value_counts().sort_index()
axes[1].bar(dist_tel.index.astype(str), dist_tel.values, color="#01B3C2")
axes[1].set_title("Distribución de Benefactores por Teléfono Registrado")
axes[1].set_xlabel("Cantidad de benefactores compartiendo el mismo teléfono")
axes[1].set_ylabel("Cantidad de teléfonos únicos")
for i, v in enumerate(dist_tel.values):
    axes[1].text(i, v + 15, str(v), ha="center", fontsize=9)

plt.tight_layout()
plt.show()

## 5. Primer Proceso ETL en Python (Extract → Transform → Load)
En esta etapa se implementa el pipeline inicial que convierte la matriz plana de 130 columnas en datos normalizados (*Silver*), aplicando:
1. **Renombramiento explícito y sin colisiones** para las 130 columnas (distinguiendo meses de 2024, 2025 y 2026).
2. **Normalización de texto y categorías** (	rato, mes_cump, municipio, 	ipo_pago, socio_categoria).
3. **Conversión híbrida de fechas** (seriales Excel 1899-12-30 + formatos DD/MM/YYYY), aislando textos inválidos.
4. **Separación de montos y notas** en ono_2026 y cuota.
5. **Generación de banderas de calidad y teléfonos compartidos** sin fusionar identidades.
6. **Transformación *Unpivot* (melt)** del histórico 2008–2026 hacia una tabla de hechos transaccional de aportes.

In [ ]:
# ==============================================================================
# 5. PROCESO ETL - TRANSFORMACIÓN Y NORMALIZACIÓN
# ==============================================================================
df_work = df_raw.copy()

# 5.1 Mapeo explícito y unívoco de las 130 columnas por posición
assert df_work.shape[1] == 130, f"Se esperaban 130 columnas y llegaron {df_work.shape[1]}"

COLS_2024_DET = [
    "2024_01_ene", "2024_02_feb", "2024_03_mar", "2024_03_bonos",
    "2024_04_abr", "2024_04_bonos", "2024_05_may", "2024_05_bonos",
    "2024_06_jun", "2024_06_bonos", "2024_07_jul", "2024_07_bonos",
    "2024_08_ago", "2024_08_bonos", "2024_09_sep", "2024_09_bonos",
    "2024_10_oct", "2024_10_bonos", "2024_11_nov", "2024_11_bonos",
    "2024_12_dic", "2024_12_bonos", "2024_banquete"
]

COLS_2025_DET = [
    "2025_01_ene", "2025_02_feb", "2025_03_mar", "2025_03_bonos",
    "2025_04_abr", "2025_04_bonos", "2025_05_may", "2025_05_bonos",
    "2025_06_jun", "2025_06_bonos", "2025_07_jul", "2025_07_bonos",
    "2025_08_ago", "2025_08_bonos", "2025_09_sep", "2025_09_bonos",
    "2025_10_oct", "2025_10_bonos", "2025_11_nov", "2025_11_bonos",
    "2025_12_dic", "2025_12_bonos"
]

COLS_2026_DET = [
    "2026_01_ene", "2026_02_feb", "2026_03_mar", "2026_03_bonos",
    "2026_04_abr", "2026_04_bonos", "2026_05_may", "2026_05_bonos",
    "2026_06_jun", "2026_06_bonos", "2026_07_jul", "2026_07_bonos",
    "2026_08_ago", "2026_08_bonos", "2026_09_sep", "2026_09_bonos",
    "2026_10_oct", "2026_10_bonos", "2026_11_nov", "2026_11_bonos",
    "2026_12_dic"
]

NOMBRES_LIMPIOS = (
    [
        "cod", "fecha_ingreso_raw", "socio_raw", "trato_raw", "nombre_completo_token",
        "nombres_token", "apellidos_token", "cedula_token", "telefono_token",
        "direccion_token", "barrio_token", "municipio_raw", "dia_cump_raw",
        "mes_cump_raw", "anio_nac_raw", "lo_atendera_raw", "mensajes_cumpleanos",
        "detalle_cumpleanos", "whatsapp_flag", "veces_en_anio_raw",
        "trim_2026_ene_mar", "trim_2026_abr_jun", "trim_2026_jul_sep", "trim_2026_oct_dic",
        "encuesta_eventos", "email_token", "certificado_donacion_raw", "donacion_extra_raw",
        "gestion_log", "llamar_bonos_2026", "calamidad", "enfermo",
        "observaciones_comentarios", "tipo_pago_raw", "recogerle_dia", "frecuencia_raw",
        "recibos_preferencia_raw", "alcancia_raw", "bono_2026_raw", "cantarilla_2026_raw",
        "rifa_virgen_carmen_raw", "cuota_raw", "ult_pago_raw", "recibos_numeros"
    ]
    + [f"total_{y}" for y in range(2008, 2024)]
    + COLS_2024_DET
    + ["total_2024"]
    + COLS_2025_DET
    + ["total_2025"]
    + COLS_2026_DET
    + ["total_2026", "acumulado_reportado"]
)

assert len(NOMBRES_LIMPIOS) == 130
df_work.columns = NOMBRES_LIMPIOS

# 5.2 Generación de ID técnico único de benefactor (nunca fusionando personas)
df_work["id_benefactor"] = range(1, len(df_work) + 1)
df_work["cod"] = pd.to_numeric(df_work["cod"], errors="coerce").astype("Int64")

# 5.3 Limpieza básica de texto y estandarización de categorías
def clean_str(val):
    if pd.isna(val):
        return np.nan
    s = re.sub(r"\s+", " ", str(val)).strip()
    return s if s != "" else np.nan

for c in [
    "nombre_completo_token", "nombres_token", "apellidos_token", "cedula_token",
    "telefono_token", "direccion_token", "barrio_token", "socio_raw", "trato_raw",
    "municipio_raw", "mes_cump_raw", "lo_atendera_raw", "tipo_pago_raw", "frecuencia_raw"
]:
    df_work[c] = df_work[c].apply(clean_str)

def normalizar_socio(val):
    if pd.isna(val):
        return "SIN_CLASIFICAR"
    v = val.upper()
    if "RETIR" in v or "NO SIGUE" in v or "NO SEG" in v:
        return "RETIRADO"
    if "BONO" in v:
        return "BONOS"
    if "PERM" in v:
        return "PERMANENTE"
    if "ESP" in v:
        return "ESPORADICO"
    if "BANQUETE" in v:
        return "BANQUETE"
    if "CANTARILLA" in v or "RIFA" in v:
        return "RIFA_CANTARILLA"
    if "ALC" in v:
        return "ALCANCIA"
    if "BINGO" in v:
        return "BINGO"
    if "NSE" in v or "TARDES" in v or "OYENTE" in v:
        return "EVENTO_NSE"
    if "EVENTO" in v or "PEREGRIN" in v or "RETIRO" in v or "EJERCICIOS" in v:
        return "EVENTO"
    return "OTRO"

df_work["socio_categoria"] = df_work["socio_raw"].apply(normalizar_socio)

VALID_TIPO_PAGO = {"DO": "DOMICILIO", "CO": "CONSIGNACION", "TR": "TRANSFERENCIA", "SE": "SEDE", "DE": "DEBITO"}
df_work["tipo_pago_norm"] = df_work["tipo_pago_raw"].map(lambda x: VALID_TIPO_PAGO.get(str(x).upper(), "OTRO_O_INVALIDO") if pd.notna(x) else np.nan)

MESES_VALIDOS = {"ENE", "FEB", "MAR", "ABR", "MAY", "JUN", "JUL", "AGO", "SEP", "OCT", "NOV", "DIC"}
df_work["mes_cump_norm"] = df_work["mes_cump_raw"].map(lambda x: x.upper() if pd.notna(x) and x.upper() in MESES_VALIDOS else np.nan)

# 5.4 Conversión híbrida de fechas (serial Excel + texto)
def parse_excel_or_text_date(val):
    if pd.isna(val):
        return pd.NaT
    if isinstance(val, (int, float, np.integer, np.floating)):
        if 20000 <= float(val) <= 60000:
            return pd.to_datetime(float(val), unit="D", origin="1899-12-30", errors="coerce")
        return pd.NaT
    if isinstance(val, (datetime, pd.Timestamp)):
        return pd.to_datetime(val)
    s = str(val).strip().replace("//", "/").replace("'", "")
    return pd.to_datetime(s, dayfirst=True, errors="coerce")

df_work["fecha_ingreso_norm"] = df_work["fecha_ingreso_raw"].apply(parse_excel_or_text_date)
df_work["ult_pago_norm"] = df_work["ult_pago_raw"].apply(parse_excel_or_text_date)
df_work["flag_fecha_ingreso_invalida"] = df_work["fecha_ingreso_raw"].notna() & df_work["fecha_ingreso_norm"].isna()
df_work["flag_ult_pago_invalido"] = df_work["ult_pago_raw"].notna() & df_work["ult_pago_norm"].isna()

# 5.5 Banderas de duplicados y teléfonos compartidos
tel_clean = df_work["telefono_token"].apply(clean_str)
tel_freq = tel_clean.value_counts(dropna=True)
df_work["personas_en_telefono"] = tel_clean.map(tel_freq).fillna(0).astype(int)
df_work["flag_telefono_compartido"] = df_work["personas_en_telefono"] > 1

ced_clean = df_work["cedula_token"].apply(clean_str)
ced_freq = ced_clean.value_counts(dropna=True)
df_work["flag_cedula_duplicada"] = ced_clean.map(ced_freq).fillna(0).astype(int) > 1

# 5.6 Separación numérico vs texto en cuota y bono_2026
df_work["cuota_valor_num"] = pd.to_numeric(df_work["cuota_raw"], errors="coerce")
df_work["bono_2026_valor_num"] = pd.to_numeric(df_work["bono_2026_raw"], errors="coerce")

# 5.7 Unpivot (Melt) de aportes históricos 2008-2026
cols_anuales_hist = [f"total_{y}" for y in range(2008, 2024)]
cols_mensuales_detalle = COLS_2024_DET + COLS_2025_DET + COLS_2026_DET

# Asegurar numéricos en todas las columnas de aportes
for c in cols_anuales_hist + cols_mensuales_detalle + ["total_2024", "total_2025", "total_2026", "acumulado_reportado"]:
    df_work[c] = pd.to_numeric(df_work[c], errors="coerce")

# Unpivot históricos 2008-2023
df_hist_long = df_work.melt(
    id_vars=["id_benefactor", "lo_atendera_raw"],
    value_vars=cols_anuales_hist,
    var_name="origen_col",
    value_name="monto"
).dropna(subset=["monto"])
df_hist_long = df_hist_long[df_hist_long["monto"] > 0].copy()
df_hist_long["anio"] = df_hist_long["origen_col"].str.extract(r"(\d{4})").astype(int)
df_hist_long["mes"] = 0
df_hist_long["recurso"] = "APORTE_ANUAL_HISTORICO"

# Unpivot detalle mensual 2024-2026
df_mensual_long = df_work.melt(
    id_vars=["id_benefactor", "lo_atendera_raw"],
    value_vars=cols_mensuales_detalle,
    var_name="origen_col",
    value_name="monto"
).dropna(subset=["monto"])
df_mensual_long = df_mensual_long[df_mensual_long["monto"] > 0].copy()
df_mensual_long["anio"] = df_mensual_long["origen_col"].str.slice(0, 4).astype(int)
df_mensual_long["mes"] = df_mensual_long["origen_col"].str.extract(r"^\d{4}_(\d{2})_")[0].fillna(12).astype(int)

def mapear_recurso(col_name: str) -> str:
    if "bonos" in col_name:
        return "BONO"
    if "banquete" in col_name:
        return "BANQUETE_NAVIDENO"
    return "APORTE_MENSUAL_REGULAR"

df_mensual_long["recurso"] = df_mensual_long["origen_col"].apply(mapear_recurso)

df_aportes_long = pd.concat([df_hist_long, df_mensual_long], ignore_index=True)
df_aportes_long["id_aporte"] = range(1, len(df_aportes_long) + 1)

print("Resumen ETL:")
print(f" - Benefactores (Silver): {len(df_work)} filas")
print(f" - Aportes normalizados >  (2008-2026): {len(df_aportes_long)} transacciones")
print(f" - Benefactores en teléfonos compartidos: {int(df_work['flag_telefono_compartido'].sum())}")
print(f" - Fechas de ingreso inválidas aisladas: {int(df_work['flag_fecha_ingreso_invalida'].sum())}")
display(df_work["socio_categoria"].value_counts().to_frame("benefactores"))

## 6. Carga al Data Lake (Parquet) y Data Warehouse Inicial (DuckDB)
Se persisten las capas **Bronze** y **Silver** en formato columnar **Parquet** dentro del Data Lake, y se crea la base analítica inicial en **DuckDB** (undacion_1380_dw.duckdb) con esquema estrella (dim_benefactor, dim_telemercaderista, dim_recurso, act_aportes) para validar consultas SQL de negocio.

In [ ]:
# ==============================================================================
# 6. CARGA A DATA LAKE (PARQUET) Y DATA WAREHOUSE (DUCKDB)
# ==============================================================================
# 6.1 Data Lake - Bronze (datos crudos preservados en formato columnar)
df_bronze = df_raw.astype(str).replace("nan", np.nan)
df_bronze.to_parquet(BRONZE_DIR / "f1380_crudo.parquet", index=False)

# 6.2 Data Lake - Silver (tablas limpias)
cols_dim_benefactor = [
    "id_benefactor", "cod", "nombre_completo_token", "nombres_token", "apellidos_token",
    "cedula_token", "telefono_token", "personas_en_telefono", "flag_telefono_compartido",
    "flag_cedula_duplicada", "direccion_token", "barrio_token", "municipio_raw",
    "trato_raw", "socio_categoria", "socio_raw", "lo_atendera_raw",
    "fecha_ingreso_norm", "ult_pago_norm", "flag_fecha_ingreso_invalida", "flag_ult_pago_invalido",
    "tipo_pago_norm", "frecuencia_raw", "cuota_valor_num", "bono_2026_valor_num",
    "mes_cump_norm", "total_2024", "total_2025", "total_2026", "acumulado_reportado"
]
df_benefactores_silver = df_work[cols_dim_benefactor].copy()
df_benefactores_silver.to_parquet(SILVER_DIR / "benefactores_limpio.parquet", index=False)
df_aportes_long.to_parquet(SILVER_DIR / "aportes_normalizados.parquet", index=False)

# 6.3 Data Warehouse en DuckDB
dw_path = DW_DIR / "fundacion_1380_dw.duckdb"
if dw_path.exists():
    dw_path.unlink()

con = duckdb.connect(str(dw_path))

con.execute("CREATE TABLE dim_benefactor AS SELECT * FROM df_benefactores_silver;")
con.execute("""
CREATE TABLE dim_telemercaderista AS
SELECT
    ROW_NUMBER() OVER (ORDER BY nombre_responsable) AS id_telemercaderista,
    nombre_responsable
FROM (
    SELECT DISTINCT COALESCE(lo_atendera_raw, 'SIN_ASIGNAR') AS nombre_responsable
    FROM df_benefactores_silver
);
""")
con.execute("""
CREATE TABLE dim_recurso AS
SELECT
    ROW_NUMBER() OVER (ORDER BY recurso) AS id_recurso,
    recurso AS nombre_recurso
FROM (SELECT DISTINCT recurso FROM df_aportes_long);
""")
con.execute("""
CREATE TABLE fact_aportes AS
SELECT
    a.id_aporte,
    a.id_benefactor,
    t.id_telemercaderista,
    r.id_recurso,
    a.anio,
    a.mes,
    a.monto,
    a.origen_col
FROM df_aportes_long a
LEFT JOIN dim_telemercaderista t
    ON COALESCE(a.lo_atendera_raw, 'SIN_ASIGNAR') = t.nombre_responsable
LEFT JOIN dim_recurso r
    ON a.recurso = r.nombre_recurso;
""")

print("Tablas creadas en DuckDB:", con.execute("SHOW TABLES").fetchall())

print("\n=== VALIDACIÓN SQL 1: RECAUDO Y TRANSACCIONES POR AÑO (TOP 10 RECIENTES) ===")
display(con.execute("""
SELECT
    anio,
    COUNT(*) AS transacciones,
    COUNT(DISTINCT id_benefactor) AS benefactores_activos,
    ROUND(SUM(monto), 0) AS total_recaudado
FROM fact_aportes
GROUP BY anio
ORDER BY anio DESC
LIMIT 10;
""").df())

print("\n=== VALIDACIÓN SQL 2: TOP 10 TELEMERCADERISTAS POR BENEFACTORES ASIGNADOS ===")
display(con.execute("""
SELECT
    COALESCE(lo_atendera_raw, 'SIN_ASIGNAR') AS responsable,
    COUNT(*) AS total_benefactores,
    SUM(CASE WHEN flag_telefono_compartido THEN 1 ELSE 0 END) AS en_tel_compartido
FROM dim_benefactor
GROUP BY 1
ORDER BY 2 DESC
LIMIT 10;
""").df())

print("\n=== VALIDACIÓN SQL 3: MUESTRA DE TELÉFONOS COMPARTIDOS (CASO ANTI-SATURACIÓN) ===")
display(con.execute("""
SELECT
    telefono_token,
    COUNT(*) AS personas_vinculadas,
    STRING_AGG(DISTINCT COALESCE(lo_atendera_raw, 'SIN_ASIGNAR'), ' | ') AS responsables_asignados
FROM dim_benefactor
WHERE flag_telefono_compartido = TRUE
GROUP BY telefono_token
ORDER BY personas_vinculadas DESC
LIMIT 10;
""").df())

con.close()
print("\nPipeline de Avance 1 ejecutado exitosamente.")